In [1]:
import json, gc
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
DATA = PROJECT / "data"
CHECKPOINTS = PROJECT / "checkpoints"

ADAPTER_1P5B_V2 = CHECKPOINTS / "qwen1p5b-qlora-v2_smaller" / "checkpoint-final"
ADAPTER_3B_V2   = CHECKPOINTS / "qwen3b-qlora-v2" / "checkpoint-final"


SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""


ADVERSARIAL_TWO_OPT = [
    ("works great",    "*"),
    ("looks nice",     "*"),
    ("hmm",            "*"),
    ("okay",           "*"),
    ("meh",            "*"),
    ("not sure",       "*"),
    ("dunno really",   "*"),
    ("cool",           "*"),
    ("sure whatever",  "*"),
    ("interesting",    "*"),
    ("i guess",        "*"),
    ("perfect",        "*"),
    ("nice one",       "*"),
    ("go with it",     "*"),
    ("that works",     "*"),
    ("A",              ["A"]),
    ("B",              ["B"]),
    ("the left one",   ["A"]),
    ("the right one",  ["B"]),
    ("neither",        []),
    ("none of them",   []),
    ("both are fine",  ["A", "B"]),
    ("A and B both work", ["A", "B"]),
]


def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]


def build_model(base_model_id, adapter_path):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(base_model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb, device_map="cuda",
    )
    m = PeftModel.from_pretrained(base, str(adapter_path))
    m.eval()
    return m, tok


def parse(model, tok, utterance, options):
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return None, raw
    return parsed, raw


def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False


def eval_test_set(model, tok, test_data, label):
    correct = 0
    per_cat = {}
    failures = []
    for i, ex in enumerate(test_data):
        if i % 20 == 0:
            print(f"  [{label}] test progress: {i}/{len(test_data)}", flush=True)
        parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
        ok = labels_equal(parsed, ex["label"])
        cat = ex.get("category", "unknown")
        per_cat.setdefault(cat, [0, 0])
        per_cat[cat][1] += 1
        if ok:
            correct += 1
            per_cat[cat][0] += 1
        else:
            failures.append((ex, parsed, raw))
    total = len(test_data)
    print(f"\n=== {label}: standard test set ===")
    print(f"overall: {correct}/{total} = {100*correct/total:.1f}%")
    for cat in sorted(per_cat):
        c, t = per_cat[cat]
        print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
    if failures:
        print(f"failures ({len(failures)}):")
        for ex, parsed, raw in failures[:15]:
            print(f"  utt={ex['utterance']!r}")
            print(f"    gold={ex['label']!r}  got={parsed!r}  raw={raw!r}")


def eval_adversarial(model, tok, label):
    print(f"\n=== {label}: 2-option adversarial ===")
    n_ambig = n_correct_ambig = n_sanity = n_correct_sanity = 0
    for utt, gold in ADVERSARIAL_TWO_OPT:
        parsed, raw = parse(model, tok, utt, ("A", "B"))
        ok = labels_equal(parsed, gold)
        is_ambig = (gold == "*")
        if is_ambig:
            n_ambig += 1
            if ok:
                n_correct_ambig += 1
        else:
            n_sanity += 1
            if ok:
                n_correct_sanity += 1
        mark = "OK  " if ok else "FAIL"
        tag = "AMB" if is_ambig else "SAN"
        print(f"  [{mark}] [{tag}] {utt!r:22s} gold={gold!r:14s} got={parsed!r}")
    print(f"\nambiguous:  {n_correct_ambig}/{n_ambig} = {100*n_correct_ambig/n_ambig:.1f}%")
    print(f"sanity:     {n_correct_sanity}/{n_sanity} = {100*n_correct_sanity/n_sanity:.1f}%")


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup (GB): {torch.cuda.memory_allocated() / 1e9:.2f}", flush=True)


# --- run 1.5B v2 only ---
test_data = load_jsonl(DATA / "test.jsonl")
print(f"loaded {len(test_data)} test examples")

print(f"\n{'='*60}\nloading 1.5B-v2...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-1.5B-Instruct", ADAPTER_1P5B_V2)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}", flush=True)

eval_test_set(model, tok, test_data, "1.5B-v2")
eval_adversarial(model, tok, "1.5B-v2")

del model, tok
free_gpu()

loaded 210 test examples

loading 1.5B-v2...


W0902 19:23:36.018000 29608 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM after load (GB): 1.23
  [1.5B-v2] test progress: 0/210


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [1.5B-v2] test progress: 20/210
  [1.5B-v2] test progress: 40/210
  [1.5B-v2] test progress: 60/210
  [1.5B-v2] test progress: 80/210
  [1.5B-v2] test progress: 100/210
  [1.5B-v2] test progress: 120/210
  [1.5B-v2] test progress: 140/210
  [1.5B-v2] test progress: 160/210
  [1.5B-v2] test progress: 180/210
  [1.5B-v2] test progress: 200/210

=== 1.5B-v2: standard test set ===
overall: 207/210 = 98.6%
  comparative          25/26 = 96.2%
  mixed_sentiment      27/28 = 96.4%
  multi_positive       12/13 = 92.3%
  negation             61/61 = 100.0%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          22/22 = 100.0%
failures (3):
  utt='B leaves D in the dust'
    gold=['B']  got=['A', 'C']  raw='["A", "C"]'
  utt='A perfect, B terrible, C okay, D terrible'
    gold=['A', 'C']  got=['A']  raw='["A"]'
  utt='A is okay but B is closer'
    gold=['A', 'B']  got=['B']  raw='["B"]'

=== 1.5B-v2: 2-option adver

In [2]:
free_gpu()

print(f"\n{'='*60}\nloading 3B-v2...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V2)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}", flush=True)

eval_test_set(model, tok, test_data, "3B-v2")
eval_adversarial(model, tok, "3B-v2")

del model, tok
free_gpu()

VRAM after cleanup (GB): 0.01

loading 3B-v2...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM after load (GB): 2.19
  [3B-v2] test progress: 0/210
  [3B-v2] test progress: 20/210
  [3B-v2] test progress: 40/210
  [3B-v2] test progress: 60/210
  [3B-v2] test progress: 80/210
  [3B-v2] test progress: 100/210
  [3B-v2] test progress: 120/210
  [3B-v2] test progress: 140/210
  [3B-v2] test progress: 160/210
  [3B-v2] test progress: 180/210
  [3B-v2] test progress: 200/210

=== 3B-v2: standard test set ===
overall: 209/210 = 99.5%
  comparative          26/26 = 100.0%
  mixed_sentiment      28/28 = 100.0%
  multi_positive       12/13 = 92.3%
  negation             61/61 = 100.0%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          22/22 = 100.0%
failures (1):
  utt='A is okay but B is closer'
    gold=['A', 'B']  got=['B']  raw='["B"]'

=== 3B-v2: 2-option adversarial ===
  [FAIL] [AMB] 'works great'          gold='*'            got=['A', 'B']
  [OK  ] [AMB] 'looks nice'           gold='*'        

In [3]:
free_gpu()

# New rule added at the end. Explicit about the failure mode:
# short affirmative words with no option referent must output "*".
SYSTEM_PROMPT_V2 = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is a short positive reaction that does NOT name any specific option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough — the user must actually reference an option (by letter, position, or description) for it to count as an endorsement.

Output ONLY the JSON. No explanation, no prose."""


# Parse using the new prompt (identical to parse() but with SYSTEM_PROMPT_V2)
def parse_v2(model, tok, utterance, options):
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V2},
        {"role": "user", "content": user_msg},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return None, raw
    return parsed, raw


# Load 3B v2 and eval adversarial + full test set with new prompt
print(f"\n{'='*60}\nloading 3B-v2 (testing new prompt)...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V2)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}", flush=True)


# --- adversarial eval with new prompt ---
print(f"\n=== 3B-v2 + new prompt: 2-option adversarial ===")
n_ambig = n_correct_ambig = n_sanity = n_correct_sanity = 0
for utt, gold in ADVERSARIAL_TWO_OPT:
    parsed, raw = parse_v2(model, tok, utt, ("A", "B"))
    ok = labels_equal(parsed, gold)
    is_ambig = (gold == "*")
    if is_ambig:
        n_ambig += 1
        if ok:
            n_correct_ambig += 1
    else:
        n_sanity += 1
        if ok:
            n_correct_sanity += 1
    mark = "OK  " if ok else "FAIL"
    tag = "AMB" if is_ambig else "SAN"
    print(f"  [{mark}] [{tag}] {utt!r:22s} gold={gold!r:14s} got={parsed!r}")
print(f"\nambiguous:  {n_correct_ambig}/{n_ambig} = {100*n_correct_ambig/n_ambig:.1f}%")
print(f"sanity:     {n_correct_sanity}/{n_sanity} = {100*n_correct_sanity/n_sanity:.1f}%")


# --- also check test set to make sure the new prompt didn't break anything ---
print(f"\n=== 3B-v2 + new prompt: full test set (regression check) ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  test progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse_v2(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

del model, tok
free_gpu()

VRAM after cleanup (GB): 0.01

loading 3B-v2 (testing new prompt)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM after load (GB): 2.19

=== 3B-v2 + new prompt: 2-option adversarial ===
  [OK  ] [AMB] 'works great'          gold='*'            got='*'
  [OK  ] [AMB] 'looks nice'           gold='*'            got='*'
  [OK  ] [AMB] 'hmm'                  gold='*'            got='*'
  [OK  ] [AMB] 'okay'                 gold='*'            got='*'
  [OK  ] [AMB] 'meh'                  gold='*'            got='*'
  [OK  ] [AMB] 'not sure'             gold='*'            got='*'
  [OK  ] [AMB] 'dunno really'         gold='*'            got='*'
  [OK  ] [AMB] 'cool'                 gold='*'            got='*'
  [OK  ] [AMB] 'sure whatever'        gold='*'            got='*'
  [OK  ] [AMB] 'interesting'          gold='*'            got='*'
  [OK  ] [AMB] 'i guess'              gold='*'            got='*'
  [OK  ] [AMB] 'perfect'              gold='*'            got='*'
  [OK  ] [AMB] 'nice one'             gold='*'            got='*'
  [OK  ] [AMB] 'go with it'           gold='*'            got='*'

In [4]:
free_gpu()

SYSTEM_PROMPT_V3 = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough. To count as an endorsement, the user must reference the options — either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""


def parse_v3(model, tok, utterance, options):
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V3},
        {"role": "user", "content": user_msg},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return None, raw
    return parsed, raw


print(f"\n{'='*60}\nloading 3B-v2 (testing refined prompt v3)...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V2)
print(f"VRAM after load (GB): {torch.cuda.memory_allocated() / 1e9:.2f}", flush=True)


# adversarial
print(f"\n=== 3B-v2 + prompt v3: 2-option adversarial ===")
n_ambig = n_correct_ambig = n_sanity = n_correct_sanity = 0
for utt, gold in ADVERSARIAL_TWO_OPT:
    parsed, raw = parse_v3(model, tok, utt, ("A", "B"))
    ok = labels_equal(parsed, gold)
    is_ambig = (gold == "*")
    if is_ambig:
        n_ambig += 1
        if ok: n_correct_ambig += 1
    else:
        n_sanity += 1
        if ok: n_correct_sanity += 1
    mark = "OK  " if ok else "FAIL"
    tag = "AMB" if is_ambig else "SAN"
    print(f"  [{mark}] [{tag}] {utt!r:22s} gold={gold!r:14s} got={parsed!r}")
print(f"\nambiguous:  {n_correct_ambig}/{n_ambig} = {100*n_correct_ambig/n_ambig:.1f}%")
print(f"sanity:     {n_correct_sanity}/{n_sanity} = {100*n_correct_sanity/n_sanity:.1f}%")


# full test set regression check
print(f"\n=== 3B-v2 + prompt v3: full test set ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  test progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse_v3(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

del model, tok
free_gpu()

VRAM after cleanup (GB): 0.01

loading 3B-v2 (testing refined prompt v3)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM after load (GB): 2.19

=== 3B-v2 + prompt v3: 2-option adversarial ===
  [OK  ] [AMB] 'works great'          gold='*'            got='*'
  [OK  ] [AMB] 'looks nice'           gold='*'            got='*'
  [OK  ] [AMB] 'hmm'                  gold='*'            got='*'
  [OK  ] [AMB] 'okay'                 gold='*'            got='*'
  [OK  ] [AMB] 'meh'                  gold='*'            got='*'
  [OK  ] [AMB] 'not sure'             gold='*'            got='*'
  [OK  ] [AMB] 'dunno really'         gold='*'            got='*'
  [OK  ] [AMB] 'cool'                 gold='*'            got='*'
  [OK  ] [AMB] 'sure whatever'        gold='*'            got='*'
  [OK  ] [AMB] 'interesting'          gold='*'            got='*'
  [OK  ] [AMB] 'i guess'              gold='*'            got='*'
  [OK  ] [AMB] 'perfect'              gold='*'            got='*'
  [OK  ] [AMB] 'nice one'             gold='*'            got='*'
  [OK  ] [AMB] 'go with it'           gold='*'            got='*'


In [5]:
# Upload both v2 adapters to HuggingFace Hub.
# Preconditions: `hf auth login` already done in this venv (session 5), token has write scope.
# Adapters go to Nightshade2304/qwen{1.5b,3b}-comparison-parser-v2, both private.

from huggingface_hub import HfApi, create_repo

api = HfApi()
HF_USER = "Nightshade2304"

UPLOADS = [
    ("1.5B", CHECKPOINTS / "qwen1p5b-qlora-v2_smaller" / "checkpoint-final",
     f"{HF_USER}/qwen1.5b-comparison-parser-v2"),
    ("3B",   CHECKPOINTS / "qwen3b-qlora-v2" / "checkpoint-final",
     f"{HF_USER}/qwen3b-comparison-parser-v2"),
]

for label, adapter_dir, repo_id in UPLOADS:
    print(f"\n{'='*60}\nuploading {label} -> {repo_id}\n{'='*60}", flush=True)
    assert adapter_dir.exists(), f"missing: {adapter_dir}"

    # create the repo (private, model type). exist_ok=True so re-runs don't fail.
    create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
    print(f"repo ready: https://huggingface.co/{repo_id}")

    # upload the whole adapter folder
    api.upload_folder(
        folder_path=str(adapter_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"v2 adapter — retrained with 173 new examples "
                       f"(ambient_positive, ambient_negative, position_words, "
                       f"multi_positive_extra); requires SYSTEM_PROMPT_V3 at inference",
    )
    print(f"uploaded {label}")

print("\nboth uploads done.")


uploading 1.5B -> Nightshade2304/qwen1.5b-comparison-parser-v2
repo ready: https://huggingface.co/Nightshade2304/qwen1.5b-comparison-parser-v2
uploaded 1.5B

uploading 3B -> Nightshade2304/qwen3b-comparison-parser-v2
repo ready: https://huggingface.co/Nightshade2304/qwen3b-comparison-parser-v2
uploaded 3B

both uploads done.
